# Modelo de regresión para predecir el puntaje en el examen final

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score, accuracy_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, chi2, f_regression, mutual_info_regression
from sklearn.linear_model import LinearRegression, LogisticRegression, Lars, Ridge, Lasso, ElasticNet, BayesianRidge, SGDRegressor, SGDClassifier
from sklearn.impute import SimpleImputer
from scipy.stats import ks_2samp
from sklearn.feature_selection import VarianceThreshold
from varclushi import VarClusHi
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
pd.set_option('display.float_format', lambda x: "{:,.2f}".format(x))

In [ ]:
ruta= os.path.join('/Users/whiz/Downloads/', 'enhanced_student_habits_performance_dataset.csv')

In [ ]:
df = pd.read_csv(ruta, sep=',')

In [ ]:
df.shape

(80000, 31)

In [ ]:
df.head(5)

,student_id,age,gender,major,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,...,screen_time,study_environment,access_to_tutoring,family_income_range,parental_support_level,motivation_level,exam_anxiety_score,learning_style,time_management_score,exam_score
0,100000,26,Male,Computer Science,7.65,3.00,0.10,Yes,70.30,6.20,...,10.90,Co-Learning Group,Yes,High,9,7,8,Reading,3.00,100
1,100001,28,Male,Arts,5.70,0.50,0.40,No,88.40,7.20,...,8.30,Co-Learning Group,Yes,Low,7,2,10,Reading,6.00,99
2,100002,17,Male,Arts,2.40,4.20,0.70,No,82.10,9.20,...,8.00,Library,Yes,High,3,9,6,Kinesthetic,7.60,98
3,100003,27,Other,Psychology,3.40,4.60,2.30,Yes,79.30,4.20,...,11.70,Co-Learning Group,Yes,Low,5,3,10,Reading,3.20,100
4,100004,25,Female,Business,4.70,0.80,2.70,Yes,62.90,6.50,...,9.40,Quiet Room,Yes,Medium,9,1,10,Reading,7.10,98


In [ ]:
df.describe([0.01, 0.99])

,student_id,age,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,exercise_frequency,mental_health_rating,previous_gpa,semester,stress_level,social_activity,screen_time,parental_support_level,motivation_level,exam_anxiety_score,time_management_score,exam_score
count,"80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00"
mean,"139,999.50",22.00,4.17,2.50,2.00,69.97,7.02,3.52,6.80,3.60,4.50,5.01,2.50,9.67,5.48,5.49,8.51,5.50,89.14
std,"23,094.16",3.75,2.00,1.45,1.16,17.33,1.47,2.29,1.92,0.46,2.30,1.95,1.70,2.78,2.87,2.87,1.80,2.60,11.59
min,"100,000.00",16.00,0.00,0.00,0.00,40.00,4.00,0.00,1.00,1.64,1.00,1.00,0.00,0.30,1.00,1.00,5.00,1.00,36.00
1%,"100,799.99",16.00,0.00,0.10,0.00,40.60,4.00,0.00,2.10,2.37,1.00,1.00,0.00,3.40,1.00,1.00,5.00,1.10,57.00
50%,"139,999.50",22.00,4.13,2.50,2.00,69.90,7.00,4.00,6.90,3.79,5.00,5.00,2.00,9.70,5.00,5.00,10.00,5.50,93.00
99%,"179,199.01",28.00,8.90,5.00,4.00,99.40,10.50,7.00,10.00,4.00,8.00,9.70,5.00,16.10,10.00,10.00,10.00,9.90,100.00
max,"179,999.00",28.00,12.00,5.00,4.00,100.00,12.00,7.00,10.00,4.00,8.00,10.00,5.00,21.00,10.00,10.00,10.00,10.00,100.00


In [ ]:
df.dtypes

student_id                         int64
age                                int64
gender                            object
major                             object
study_hours_per_day              float64
social_media_hours               float64
netflix_hours                    float64
part_time_job                     object
attendance_percentage            float64
sleep_hours                      float64
diet_quality                      object
exercise_frequency                 int64
parental_education_level          object
internet_quality                  object
mental_health_rating             float64
extracurricular_participation     object
previous_gpa                     float64
semester                           int64
stress_level                     float64
dropout_risk                      object
social_activity                    int64
screen_time                      float64
study_environment                 object
access_to_tutoring                object
family_income_ra

In [ ]:
df[['access_to_tutoring']].value_counts()

access_to_tutoring
No                    40039
Yes                   39961
Name: count, dtype: int64

### Variables

In [ ]:
um = ['student_id']
tgt = ['exam_score']
tgt_dis = ['learning_style']

In [ ]:
cols_num = []
cols_str = []

In [ ]:
cols_num = df.select_dtypes( include = ['float'] ).columns.tolist()
cols_num = cols_num + df.select_dtypes( include = ['int'] ).columns.tolist()

In [ ]:
cols_str = df.select_dtypes( include = ['object'] ).columns.tolist()

In [ ]:
df.shape

(80000, 31)

In [ ]:
cols_num = [c for c in cols_num if c not in tgt+um]
cols_str = [c for c in cols_str if c not in tgt+um]

In [ ]:
len(cols_num) + len(cols_str) + 2 # ese dos es um y tgt

31

In [ ]:
df_final = df[ um + cols_num + cols_str + tgt ]

In [ ]:
df_final.sample(1)

,student_id,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,...,diet_quality,parental_education_level,internet_quality,extracurricular_participation,dropout_risk,study_environment,access_to_tutoring,family_income_range,learning_style,exam_score
28343,128343,0.00,4.70,1.30,85.10,4.00,4.50,3.85,5.20,7.00,...,Fair,Bachelor,Low,Yes,No,Library,Yes,High,Visual,97


# Ingeniería de Variables

In [ ]:
df_final['horas_de_ocio'] = df_final['social_media_hours'] + df_final['netflix_hours']
##df_final['ratio_estudio_ocio'] = df_final['study_hours_per_day'] / (df_final['horas_de_ocio']).replace([np.inf, -np.inf, np.nan], 0)
df_final['actividad_total_diaria'] = df_final['study_hours_per_day'] + df_final['horas_de_ocio'] + df_final['sleep_hours']
df_final['manejo_de'] = df_final['time_management_score'] * df_final['study_hours_per_day']
df_final['ratio_productividad_en_pantalla'] = (df_final['screen_time'] - df_final['social_media_hours'] - df_final['netflix_hours']) / (df_final['screen_time'] + 1)
df_final['estres_ansiedad'] = df_final['stress_level'] + df_final['exam_anxiety_score']
df_final['manejo_de_estres'] = df_final['exam_anxiety_score'] - df_final['motivation_level']
df_final['acceso_a_tutor_numero'] = df_final['access_to_tutoring'].map({'Yes': 1, 'No': 0})
df_final['ratio_acceso_apoyo'] = df_final['parental_support_level'] / (df_final['acceso_a_tutor_numero'] + 1)
df_final['aprovecamiento'] = df_final['previous_gpa'] * df_final['attendance_percentage']

/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_15831/1286370015.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['horas_de_ocio'] = df_final['social_media_hours'] + df_final['netflix_hours']
/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_15831/1286370015.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['actividad_total_diaria'] = df_final['study_hours_per_day'] + df_final['horas_de_ocio'] + df_final['sleep_hours']
/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn

In [ ]:
df_final[['horas_de_ocio']].value_counts()

horas_de_ocio
5.00             1656
4.50             1579
4.00             1558
4.10             1446
4.40             1438
                 ... 
8.40               39
0.10               34
8.60               31
9.00               14
0.00               13
Name: count, Length: 153, dtype: int64

In [ ]:
df_final.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['student_id', 'study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'attendance_percentage', 'sleep_hours',
       'mental_health_rating', 'previous_gpa', 'stress_level', 'screen_time',
       'time_management_score', 'age', 'exercise_frequency', 'semester',
       'social_activity', 'parental_support_level', 'motivation_level',
       'exam_anxiety_score', 'gender', 'major', 'part_time_job',
       'diet_quality', 'parental_education_level', 'internet_quality',
       'extracurricular_participation', 'dropout_risk', 'study_environment',
       'access_to_tutoring', 'family_income_range', 'learning_style',
       'exam_score', 'horas_de_ocio', 'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'acceso_a_tutor_numero', 'ratio_acceso_apoyo',
       'aprovecamiento'],
      dtype='object')>

In [ ]:
cols_num = cols_num + ['horas_de_ocio',
       'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'acceso_a_tutor_numero', 'ratio_acceso_apoyo',
       'aprovecamiento']

In [ ]:
df_final.shape

(80000, 40)

In [ ]:
len(cols_num),len(cols_str)

(26, 12)

# Análisis Exploratorio

## Categóricas

In [ ]:
### La falta de información también es información para las categóricas
for c in cols_str:
    df_final[c] = df_final[c].fillna("Sin categoría")

/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_15831/2542936986.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final[c] = df_final[c].fillna("Sin categoría")


## Análisis de Frecuencias

In [ ]:
def freq(df, var):
    if type(var) != list:
        var = [var]
    for v in var:
        aux = df[v].value_counts().to_frame().rename(columns={'count':'FA'})
        aux['FR'] = aux['FA'] / aux['FA'].sum()
        aux[['FAA','FRA']] = aux.apply( np.cumsum )
        print(f'Tabla de frecuencias para la variable {v} \n')
        print(aux,'\n')

In [ ]:
for c in cols_str:
    freq( df_final, c )

Tabla de frecuencias para la variable gender 

           FA   FR    FAA  FRA
gender                        
Female  26705 0.33  26705 0.33
Male    26698 0.33  53403 0.67
Other   26597 0.33  80000 1.00 

Tabla de frecuencias para la variable major 

                     FA   FR    FAA  FRA
major                                   
Arts              13505 0.17  13505 0.17
Psychology        13437 0.17  26942 0.34
Computer Science  13352 0.17  40294 0.50
Business          13276 0.17  53570 0.67
Engineering       13229 0.17  66799 0.83
Biology           13201 0.17  80000 1.00 

Tabla de frecuencias para la variable part_time_job 

                  FA   FR    FAA  FRA
part_time_job                        
No             40195 0.50  40195 0.50
Yes            39805 0.50  80000 1.00 

Tabla de frecuencias para la variable diet_quality 

                 FA   FR    FAA  FRA
diet_quality                        
Good          39935 0.50  39935 0.50
Fair          26713 0.33  66648 0.83
Poor       

## Normalización

In [ ]:
def norm(df, v, umbral):
    aux = df[v].value_counts(True).to_frame()
    aux[f'n_{v}'] = np.where( aux['proportion'] < umbral , 'CAT_PEQUEÑAS', aux.index )
    valor = aux.head(1)[f'n_{v}'].values[0]
    if aux.loc[aux[f'n_{v}'] == 'CAT_PEQUEÑAS']['proportion'].sum() < umbral:
        aux[f'n_{v}'].replace( {'CAT_PEQUEÑAS':valor} , inplace=True )
    aux.drop('proportion',axis=1, inplace=True)
    aux.reset_index(inplace=True)

    return df.merge(aux, left_on=[v], right_on = [v], how='left').drop(v,axis=1)

In [ ]:
for c in cols_str:
    df_final = norm( df_final, c,  0.03 )

In [ ]:
cols_norm = [c for c in df_final.columns.tolist() if c[:2] == 'n_']

In [ ]:
for c in cols_norm:
    freq( df_final, c )

Tabla de frecuencias para la variable n_gender 

             FA   FR    FAA  FRA
n_gender                        
Female    26705 0.33  26705 0.33
Male      26698 0.33  53403 0.67
Other     26597 0.33  80000 1.00 

Tabla de frecuencias para la variable n_major 

                     FA   FR    FAA  FRA
n_major                                 
Arts              13505 0.17  13505 0.17
Psychology        13437 0.17  26942 0.34
Computer Science  13352 0.17  40294 0.50
Business          13276 0.17  53570 0.67
Engineering       13229 0.17  66799 0.83
Biology           13201 0.17  80000 1.00 

Tabla de frecuencias para la variable n_part_time_job 

                    FA   FR    FAA  FRA
n_part_time_job                        
No               40195 0.50  40195 0.50
Yes              39805 0.50  80000 1.00 

Tabla de frecuencias para la variable n_diet_quality 

                   FA   FR    FAA  FRA
n_diet_quality                        
Good            39935 0.50  39935 0.50
Fair            

In [ ]:
len(cols_norm),cols_norm

(12,
 ['n_gender',
  'n_major',
  'n_part_time_job',
  'n_diet_quality',
  'n_parental_education_level',
  'n_internet_quality',
  'n_extracurricular_participation',
  'n_dropout_risk',
  'n_study_environment',
  'n_access_to_tutoring',
  'n_family_income_range',
  'n_learning_style'])

In [ ]:
df_final.shape

(80000, 40)

## Unarias

In [ ]:
unarias = [ v for v, conteo in zip( cols_norm  , [df_final[v2].unique().shape[0] for v2 in cols_norm ] ) if conteo == 1 ]

In [ ]:
len(unarias), unarias

(1, ['n_dropout_risk'])

In [ ]:
len(cols_norm)

12

In [ ]:
cols_norm = [c for c in cols_norm if c not in unarias]

In [ ]:
len(cols_norm)

11

## Numéricas

In [ ]:
X = df_final[cols_num]

In [ ]:
X.shape

(80000, 26)

In [ ]:
X.head()

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,exam_anxiety_score,horas_de_ocio,actividad_total_diaria,manejo_de,ratio_productividad_en_pantalla,estres_ansiedad,manejo_de_estres,acceso_a_tutor_numero,ratio_acceso_apoyo,aprovecamiento
0,7.65,3.00,0.10,70.30,6.20,6.00,4.00,5.80,10.90,3.00,...,8,3.10,16.95,22.94,0.66,13.80,1,1,4.50,281.20
1,5.70,0.50,0.40,88.40,7.20,6.80,4.00,5.80,8.30,6.00,...,10,0.90,13.80,34.20,0.80,15.80,8,1,3.50,353.60
2,2.40,4.20,0.70,82.10,9.20,5.70,3.79,8.00,8.00,7.60,...,6,4.90,16.50,18.24,0.34,14.00,-3,1,1.50,311.16
3,3.40,4.60,2.30,79.30,4.20,8.50,4.00,4.60,11.70,3.20,...,10,6.90,14.50,10.88,0.38,14.60,7,1,2.50,317.20
4,4.70,0.80,2.70,62.90,6.50,9.20,4.00,5.70,9.40,7.10,...,10,3.50,14.70,33.37,0.57,15.70,9,1,4.50,251.60


## Análisis Univariado

In [ ]:
X.describe([0.01, 0.99])

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,exam_anxiety_score,horas_de_ocio,actividad_total_diaria,manejo_de,ratio_productividad_en_pantalla,estres_ansiedad,manejo_de_estres,acceso_a_tutor_numero,ratio_acceso_apoyo,aprovecamiento
count,"80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00",...,"80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00"
mean,4.17,2.50,2.00,69.97,7.02,6.80,3.60,5.01,9.67,5.50,...,8.51,4.50,15.69,22.96,0.48,13.52,3.02,0.50,4.12,252.08
std,2.00,1.45,1.16,17.33,1.47,1.92,0.46,1.95,2.78,2.60,...,1.80,1.85,3.09,16.30,0.14,2.65,4.57,0.50,2.67,70.83
min,0.00,0.00,0.00,40.00,4.00,1.00,1.64,1.00,0.30,1.00,...,5.00,0.00,4.80,0.00,-0.00,6.00,-5.00,0.00,0.50,78.06
1%,0.00,0.10,0.00,40.60,4.00,2.10,2.37,1.00,3.40,1.10,...,5.00,0.60,8.80,0.00,0.10,7.10,-5.00,0.00,0.50,119.92
50%,4.13,2.50,2.00,69.90,7.00,6.90,3.79,5.00,9.70,5.50,...,10.00,4.50,15.70,19.72,0.48,13.70,5.00,0.00,3.50,248.00
99%,8.90,5.00,4.00,99.40,10.50,10.00,4.00,9.70,16.10,9.90,...,10.00,8.40,22.90,69.59,0.79,19.20,9.00,1.00,10.00,394.40
max,12.00,5.00,4.00,100.00,12.00,10.00,4.00,10.00,21.00,10.00,...,10.00,9.00,28.80,113.17,0.91,20.00,9.00,1.00,10.00,400.00


In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = encoder.fit_transform(df_final[cols_norm] )

In [ ]:
encoded_data

array([[0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 1.],
       [0., 0., 1., ..., 1., 0., 0.],
       [0., 0., 1., ..., 0., 1., 0.]])

In [ ]:
new_names = encoder.get_feature_names_out(cols_norm)

In [ ]:
# Crea el DataFrame con los nombres correctos
df_encoded = pd.DataFrame(encoded_data, columns=new_names)
print(df_encoded)

       n_gender_Female  n_gender_Male  n_gender_Other  n_major_Arts  \
0                 0.00           1.00            0.00          0.00   
1                 0.00           1.00            0.00          1.00   
2                 0.00           1.00            0.00          1.00   
3                 0.00           0.00            1.00          0.00   
4                 1.00           0.00            0.00          0.00   
...                ...            ...             ...           ...   
79995             0.00           1.00            0.00          0.00   
79996             1.00           0.00            0.00          0.00   
79997             1.00           0.00            0.00          1.00   
79998             0.00           0.00            1.00          0.00   
79999             0.00           0.00            1.00          0.00   

       n_major_Biology  n_major_Business  n_major_Computer Science  \
0                 0.00              0.00                      1.00   
1      

In [ ]:
df_f = pd.concat( [df_final,df_encoded]  ,axis=1)

In [ ]:
df_f

,student_id,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,...,n_study_environment_Quiet Room,n_access_to_tutoring_No,n_access_to_tutoring_Yes,n_family_income_range_High,n_family_income_range_Low,n_family_income_range_Medium,n_learning_style_Auditory,n_learning_style_Kinesthetic,n_learning_style_Reading,n_learning_style_Visual
0,100000,7.65,3.00,0.10,70.30,6.20,6.00,4.00,5.80,10.90,...,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00
1,100001,5.70,0.50,0.40,88.40,7.20,6.80,4.00,5.80,8.30,...,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00
2,100002,2.40,4.20,0.70,82.10,9.20,5.70,3.79,8.00,8.00,...,0.00,0.00,1.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00
3,100003,3.40,4.60,2.30,79.30,4.20,8.50,4.00,4.60,11.70,...,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00
4,100004,4.70,0.80,2.70,62.90,6.50,9.20,4.00,5.70,9.40,...,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,179995,3.70,2.10,1.00,80.80,6.10,1.00,3.40,2.10,8.30,...,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00
79996,179996,1.20,0.40,2.90,99.50,4.10,5.70,2.26,3.90,4.70,...,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00
79997,179997,4.10,1.60,1.60,46.10,8.30,6.70,3.15,5.60,7.50,...,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00
79998,179998,3.80,0.60,3.50,58.70,5.80,7.60,3.67,2.40,9.30,...,1.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00


In [ ]:
df_f.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 78 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   student_id                               80000 non-null  int64  
 1   study_hours_per_day                      80000 non-null  float64
 2   social_media_hours                       80000 non-null  float64
 3   netflix_hours                            80000 non-null  float64
 4   attendance_percentage                    80000 non-null  float64
 5   sleep_hours                              80000 non-null  float64
 6   mental_health_rating                     80000 non-null  float64
 7   previous_gpa                             80000 non-null  float64
 8   stress_level                             80000 non-null  float64
 9   screen_time                              80000 non-null  float64
 10  time_management_score                    80000

In [ ]:
cols_norm

['n_gender',
 'n_major',
 'n_part_time_job',
 'n_diet_quality',
 'n_parental_education_level',
 'n_internet_quality',
 'n_extracurricular_participation',
 'n_study_environment',
 'n_access_to_tutoring',
 'n_family_income_range',
 'n_learning_style']

In [ ]:
df_f = df_f.drop(columns=['n_gender',
 'n_major',
 'n_part_time_job',
 'n_diet_quality',
 'n_parental_education_level',
 'n_internet_quality',
 'n_extracurricular_participation',
 'n_study_environment',
 'n_access_to_tutoring',
 'n_family_income_range',
 'n_learning_style','n_dropout_risk' ])

In [ ]:
df_f.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['student_id', 'study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'attendance_percentage', 'sleep_hours',
       'mental_health_rating', 'previous_gpa', 'stress_level', 'screen_time',
       'time_management_score', 'age', 'exercise_frequency', 'semester',
       'social_activity', 'parental_support_level', 'motivation_level',
       'exam_anxiety_score', 'exam_score', 'horas_de_ocio',
       'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'acceso_a_tutor_numero', 'ratio_acceso_apoyo',
       'aprovecamiento', 'n_gender_Female', 'n_gender_Male', 'n_gender_Other',
       'n_major_Arts', 'n_major_Biology', 'n_major_Business',
       'n_major_Computer Science', 'n_major_Engineering', 'n_major_Psychology',
       'n_part_time_job_No', 'n_part_time_job_Yes', 'n_diet_quality_Fair',
       'n_diet_quality_Good', 'n_diet_quality_Poor',
       'n_parental_

In [ ]:
X = df_f[['study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'attendance_percentage', 'sleep_hours',
       'mental_health_rating', 'previous_gpa', 'stress_level', 'screen_time',
       'time_management_score', 'age', 'exercise_frequency', 'semester',
       'social_activity', 'parental_support_level', 'motivation_level',
       'exam_anxiety_score', 'horas_de_ocio','actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'acceso_a_tutor_numero', 'ratio_acceso_apoyo',
       'aprovecamiento', 'n_gender_Female', 'n_gender_Male',
       'n_gender_Other', 'n_major_Arts', 'n_major_Biology', 'n_major_Business',
       'n_major_Computer Science', 'n_major_Engineering', 'n_major_Psychology',
       'n_part_time_job_No', 'n_part_time_job_Yes', 'n_diet_quality_Fair',
       'n_diet_quality_Good', 'n_diet_quality_Poor',
       'n_parental_education_level_Bachelor',
       'n_parental_education_level_High School',
       'n_parental_education_level_Master', 'n_parental_education_level_PhD',
       'n_parental_education_level_Some College', 'n_internet_quality_High',
       'n_internet_quality_Low', 'n_internet_quality_Medium',
       'n_extracurricular_participation_No',
       'n_extracurricular_participation_Yes', 'n_study_environment_Cafe',
       'n_study_environment_Co-Learning Group', 'n_study_environment_Dorm',
       'n_study_environment_Library', 'n_study_environment_Quiet Room',
       'n_access_to_tutoring_No', 'n_access_to_tutoring_Yes',
       'n_family_income_range_High', 'n_family_income_range_Low',
       'n_family_income_range_Medium', 'n_learning_style_Auditory',
       'n_learning_style_Kinesthetic', 'n_learning_style_Reading',
       'n_learning_style_Visual']]

In [ ]:
y = df_f [['exam_score']]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=777)

In [ ]:
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((56000, 64), (56000, 1), (24000, 64), (24000, 1))

In [ ]:
X_train.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'mental_health_rating',
       'previous_gpa', 'stress_level', 'screen_time', 'time_management_score',
       'age', 'exercise_frequency', 'semester', 'social_activity',
       'parental_support_level', 'motivation_level', 'exam_anxiety_score',
       'horas_de_ocio', 'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'acceso_a_tutor_numero', 'ratio_acceso_apoyo',
       'aprovecamiento', 'n_gender_Female', 'n_gender_Male', 'n_gender_Other',
       'n_major_Arts', 'n_major_Biology', 'n_major_Business',
       'n_major_Computer Science', 'n_major_Engineering', 'n_major_Psychology',
       'n_part_time_job_No', 'n_part_time_job_Yes', 'n_diet_quality_Fair',
       'n_diet_quality_Good', 'n_diet_quality_Poor',
       'n_parental_education_level_Bachelor',
       '

In [ ]:
X_train.sample(10)

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,n_study_environment_Quiet Room,n_access_to_tutoring_No,n_access_to_tutoring_Yes,n_family_income_range_High,n_family_income_range_Low,n_family_income_range_Medium,n_learning_style_Auditory,n_learning_style_Kinesthetic,n_learning_style_Reading,n_learning_style_Visual
34022,3.40,0.10,2.70,71.90,8.10,7.40,3.96,3.30,8.10,4.90,...,1.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00
76,3.60,4.90,1.60,73.50,5.60,6.30,3.17,4.10,11.70,4.60,...,1.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00
74976,5.20,4.90,0.50,81.10,8.30,7.80,4.00,3.40,10.70,6.40,...,1.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00
1984,7.90,4.70,2.20,45.10,5.80,9.90,4.00,2.30,16.50,4.90,...,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00
42119,5.00,4.10,3.10,42.10,8.50,5.80,4.00,3.70,14.00,7.50,...,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00
31360,2.60,5.00,0.10,87.10,9.10,8.60,3.89,6.30,8.40,4.70,...,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00
24798,7.80,1.90,3.90,89.20,10.80,7.90,3.39,2.20,14.10,6.30,...,0.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00
75319,3.40,0.60,2.10,50.60,8.60,3.90,3.81,5.20,6.80,3.60,...,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00
39942,3.30,4.70,3.30,44.00,6.80,3.80,4.00,5.20,11.80,4.00,...,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00
2423,3.20,1.20,3.70,53.30,4.80,4.90,2.70,3.10,8.90,4.00,...,0.00,1.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00


### StandardScaler

In [ ]:
sc_xc = StandardScaler()
sc_yc = StandardScaler()
Xs_xc = pd.DataFrame(sc_xc.fit_transform(X_train),columns = X.columns)
Xs_yc = pd.DataFrame(sc_yc.fit_transform(y_train),columns = y.columns)

In [ ]:
Xs_xc.columns

Index(['study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'mental_health_rating',
       'previous_gpa', 'stress_level', 'screen_time', 'time_management_score',
       'age', 'exercise_frequency', 'semester', 'social_activity',
       'parental_support_level', 'motivation_level', 'exam_anxiety_score',
       'horas_de_ocio', 'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'acceso_a_tutor_numero', 'ratio_acceso_apoyo',
       'aprovecamiento', 'n_gender_Female', 'n_gender_Male', 'n_gender_Other',
       'n_major_Arts', 'n_major_Biology', 'n_major_Business',
       'n_major_Computer Science', 'n_major_Engineering', 'n_major_Psychology',
       'n_part_time_job_No', 'n_part_time_job_Yes', 'n_diet_quality_Fair',
       'n_diet_quality_Good', 'n_diet_quality_Poor',
       'n_parental_education_level_Bachelor',
       'n_parental_education_level_High School

In [ ]:
Xs_xc.head()

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,n_study_environment_Quiet Room,n_access_to_tutoring_No,n_access_to_tutoring_Yes,n_family_income_range_High,n_family_income_range_Low,n_family_income_range_Medium,n_learning_style_Auditory,n_learning_style_Kinesthetic,n_learning_style_Reading,n_learning_style_Visual
0,0.46,-0.76,1.56,0.78,-1.17,-0.94,0.86,-1.96,0.44,-0.85,...,2.00,1.00,-1.00,1.41,-0.71,-0.71,-0.58,-0.58,-0.58,1.73
1,-2.08,0.90,0.87,1.69,1.29,0.73,0.86,-0.57,-0.74,-0.04,...,2.00,-1.00,1.00,1.41,-0.71,-0.71,-0.58,-0.58,-0.58,1.73
2,1.43,1.04,0.95,-1.29,-1.10,0.83,0.86,-1.09,1.77,0.54,...,-0.50,-1.00,1.00,1.41,-0.71,-0.71,-0.58,-0.58,-0.58,1.73
3,1.21,1.59,0.09,0.82,-1.44,-0.42,-1.17,0.04,1.52,-0.88,...,-0.50,-1.00,1.00,1.41,-0.71,-0.71,1.74,-0.58,-0.58,-0.58
4,0.91,0.28,0.18,-1.53,0.19,-0.68,-0.22,1.68,1.16,-0.04,...,2.00,1.00,-1.00,-0.71,1.42,-0.71,-0.58,1.73,-0.58,-0.58


In [ ]:
Xs_yc.head()

,exam_score
0,0.94
1,0.94
2,0.85
3,-0.87
4,-0.01


## Modelado

In [ ]:
dc_scores ={}

### Regresión Lasso

In [ ]:
model = Lasso()

In [ ]:
model.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'max_iter': 1000,
 'positive': False,
 'precompute': False,
 'random_state': None,
 'selection': 'cyclic',
 'tol': 0.0001,
 'warm_start': False}

In [ ]:
model.fit(Xs_xc, Xs_yc)
ls_medias = cross_val_score(estimator=model, X=Xs_xc, y = Xs_yc, cv = 4, n_jobs=-1, scoring="r2")
np.mean(ls_medias), np.std(ls_medias)

(-0.0001808033278901111, 0.00019499262214961547)

In [ ]:
#Combinación de parámetros
param_grid = {
    "alpha": [x for x in range(1, 100)] + [y/10 for y in range(10)],
    "tol": [0.00001, 0.0000001, 0.01],
    "selection": ['cyclic', 'random']
}

In [ ]:
#Espacio de hiperparámetros
np.prod(list(map(len, param_grid.values())))

654

In [ ]:
%%time
clf = GridSearchCV(model, param_grid, cv=4, error_score=-1000, n_jobs=-1, scoring="r2", verbose=5)
clf.fit(X_train, y_train)
print("Best score: " + str(clf.best_score_))

Fitting 4 folds for each of 654 candidates, totalling 2616 fits
[CV 2/4] END alpha=1, selection=cyclic, tol=1e-05;, score=0.822 total time=   0.1s
[CV 4/4] END alpha=1, selection=cyclic, tol=1e-05;, score=0.819 total time=   0.1s
[CV 2/4] END alpha=1, selection=cyclic, tol=1e-07;, score=0.822 total time=   0.2s
[CV 1/4] END alpha=1, selection=cyclic, tol=1e-07;, score=0.816 total time=   0.2s
[CV 1/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.816 total time=   0.1s
[CV 2/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.822 total time=   0.1s
[CV 3/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.819 total time=   0.1s
[CV 4/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.819 total time=   0.1s
[CV 1/4] END alpha=1, selection=random, tol=1e-05;, score=0.816 total time=   0.2s
[CV 2/4] END alpha=1, selection=random, tol=1e-05;, score=0.822 total time=   0.2s
[CV 3/4] END alpha=1, selection=random, tol=1e-05;, score=0.819 total time=   0.2s
[CV 4/4] END alpha=1, selec

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/py

[CV 1/4] END alpha=0.0, selection=cyclic, tol=1e-07;, score=0.869 total time=   3.4s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.691e+05, tolerance: 5.689e+04 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model 

[CV 1/4] END alpha=0.0, selection=cyclic, tol=0.01;, score=0.869 total time=   4.0s
[CV 3/4] END alpha=0.0, selection=cyclic, tol=0.01;, score=0.869 total time=   4.0s


...

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.073e+02, tolerance: 5.668e-01
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.965e+03, tolerance: 5.634e-01
  model = cd_fast.enet_coordinate_descent(


[CV 3/4] END alpha=0.3, selection=random, tol=1e-07;, score=0.823 total time=   2.0s
[CV 4/4] END alpha=0.3, selection=random, tol=1e-07;, score=0.822 total time=   1.9s
[CV 1/4] END alpha=0.4, selection=cyclic, tol=1e-05;, score=0.817 total time=   0.3s
[CV 2/4] END alpha=0.4, selection=cyclic, tol=1e-05;, score=0.823 total time=   0.3s
[CV 3/4] END alpha=0.4, selection=cyclic, tol=1e-05;, score=0.820 total time=   0.3s
[CV 2/4] END alpha=0.3, selection=random, tol=0.01;, score=0.828 total time=   1.2s
[CV 3/4] END alpha=0.3, selection=random, tol=0.01;, score=0.827 total time=   1.2s
[CV 4/4] END alpha=0.4, selection=cyclic, tol=1e-05;, score=0.820 total time=   0.3s
[CV 1/4] END alpha=0.4, selection=cyclic, tol=1e-07;, score=0.817 total time=   0.3s
[CV 2/4] END alpha=0.4, selection=cyclic, tol=1e-07;, score=0.823 total time=   0.3s
[CV 4/4] END alpha=0.4, selection=cyclic, tol=1e-07;, score=0.820 total time=   0.3s
[CV 2/4] END alpha=0.4, selection=cyclic, tol=0.01;, score=0.823 to

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(


Best score: 0.8697480847516432
CPU times: user 1min 5s, sys: 12.4 s, total: 1min 17s
Wall time: 1min 33s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.899e+05, tolerance: 7.539e+04 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


In [ ]:
summary = pd.DataFrame(clf.cv_results_)

In [ ]:
summary.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_selection,param_tol,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
0,0.13,0.01,0.00,0.00,1,cyclic,0.00,"{'alpha': 1, 'selection': 'cyclic', 'tol': 1e-05}",0.82,0.82,0.82,0.82,0.82,0.00,62
1,0.17,0.01,0.00,0.00,1,cyclic,0.00,"{'alpha': 1, 'selection': 'cyclic', 'tol': 1e-07}",0.82,0.82,0.82,0.82,0.82,0.00,59
2,0.11,0.01,0.00,0.00,1,cyclic,0.01,"{'alpha': 1, 'selection': 'cyclic', 'tol': 0.01}",0.82,0.82,0.82,0.82,0.82,0.00,65
3,0.21,0.02,0.00,0.00,1,random,0.00,"{'alpha': 1, 'selection': 'random', 'tol': 1e-05}",0.82,0.82,0.82,0.82,0.82,0.00,61
4,0.27,0.02,0.00,0.00,1,random,0.00,"{'alpha': 1, 'selection': 'random', 'tol': 1e-07}",0.82,0.82,0.82,0.82,0.82,0.00,60


In [ ]:
summary.sort_values(by = "rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_selection,param_tol,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
599,4.70,0.25,0.00,0.00,0.00,random,0.01,"{'alpha': 0.0, 'selection': 'random', 'tol': 0...",0.87,0.87,0.87,0.87,0.87,0.00,1
598,3.80,0.17,0.01,0.00,0.00,random,0.00,"{'alpha': 0.0, 'selection': 'random', 'tol': 1...",0.87,0.87,0.87,0.87,0.87,0.00,2
594,4.90,0.06,0.00,0.00,0.00,cyclic,0.00,"{'alpha': 0.0, 'selection': 'cyclic', 'tol': 1...",0.87,0.87,0.87,0.87,0.87,0.00,3
595,3.96,0.49,0.00,0.00,0.00,cyclic,0.00,"{'alpha': 0.0, 'selection': 'cyclic', 'tol': 1...",0.87,0.87,0.87,0.87,0.87,0.00,3
596,4.36,0.40,0.01,0.00,0.00,cyclic,0.01,"{'alpha': 0.0, 'selection': 'cyclic', 'tol': 0...",0.87,0.87,0.87,0.87,0.87,0.00,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,0.03,0.01,0.00,0.00,99,cyclic,0.01,"{'alpha': 99, 'selection': 'cyclic', 'tol': 0.01}",0.17,0.17,0.17,0.17,0.17,0.00,649
591,0.03,0.00,0.00,0.00,99,random,0.00,"{'alpha': 99, 'selection': 'random', 'tol': 1e...",0.17,0.17,0.17,0.17,0.17,0.00,649
592,0.03,0.00,0.00,0.00,99,random,0.00,"{'alpha': 99, 'selection': 'random', 'tol': 1e...",0.17,0.17,0.17,0.17,0.17,0.00,649
593,0.03,0.00,0.00,0.00,99,random,0.01,"{'alpha': 99, 'selection': 'random', 'tol': 0.01}",0.17,0.17,0.17,0.17,0.17,0.00,649


In [ ]:
dc_scores[str(model).split("(")[0]] = {"model": clf.best_estimator_, "score": clf.best_score_}

In [ ]:
dc_scores

{'Lasso': {'model': Lasso(alpha=0.0, selection='random', tol=0.01),
  'score': 0.8697480847516432}}

### Ridge Regression

In [ ]:
model = Ridge()

In [ ]:
model.fit(X_train, y_train)
ls_medias = cross_val_score(estimator=model, X=X_test, y = y_test, cv = 4, n_jobs=-1, scoring="r2")
np.mean(ls_medias), np.std(ls_medias)

(0.87108087089374, 0.0008641272099908312)

In [ ]:
model.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'max_iter': None,
 'positive': False,
 'random_state': None,
 'solver': 'auto',
 'tol': 0.0001}

In [ ]:
param_grid = {
    "alpha": [x for x in range(1, 100)] + [y/10 for y in range(10)],
    "tol": [0.0000001, 0.01],
    "solver": ['auto', 'svd', 'cholesky', 'lsqr'],
    "max_iter": [500]
}

In [ ]:
np.prod(list(map(len, param_grid.values())))

872

In [ ]:
%%time
clf = GridSearchCV(model, param_grid, cv=4, error_score=-1000, n_jobs=-1, scoring="r2")
clf.fit(X_train, y_train)
print("Best score: " + str(clf.best_score_))

/Users/whiz/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best score: 0.8697477723540181
CPU times: user 5.4 s, sys: 10.8 s, total: 16.2 s
Wall time: 1min 6s


In [ ]:
summary = pd.DataFrame(clf.cv_results_)

In [ ]:
summary.sort_values(by = "rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_max_iter,param_solver,param_tol,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
806,0.15,0.00,0.00,0.00,0.10,500,lsqr,0.00,"{'alpha': 0.1, 'max_iter': 500, 'solver': 'lsq...",0.87,0.87,0.87,0.87,0.87,0.00,1
798,0.17,0.03,0.00,0.00,0.00,500,lsqr,0.00,"{'alpha': 0.0, 'max_iter': 500, 'solver': 'lsq...",0.87,0.87,0.87,0.87,0.87,0.00,2
814,0.16,0.01,0.00,0.00,0.20,500,lsqr,0.00,"{'alpha': 0.2, 'max_iter': 500, 'solver': 'lsq...",0.87,0.87,0.87,0.87,0.87,0.00,3
804,0.03,0.00,0.00,0.00,0.10,500,cholesky,0.00,"{'alpha': 0.1, 'max_iter': 500, 'solver': 'cho...",0.87,0.87,0.87,0.87,0.87,0.00,4
801,0.02,0.00,0.00,0.00,0.10,500,auto,0.01,"{'alpha': 0.1, 'max_iter': 500, 'solver': 'aut...",0.87,0.87,0.87,0.87,0.87,0.00,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831,0.03,0.00,0.00,0.00,0.40,500,lsqr,0.01,"{'alpha': 0.4, 'max_iter': 500, 'solver': 'lsq...",0.82,0.82,0.82,0.82,0.82,0.00,868
823,0.04,0.00,0.00,0.00,0.30,500,lsqr,0.01,"{'alpha': 0.3, 'max_iter': 500, 'solver': 'lsq...",0.82,0.82,0.82,0.82,0.82,0.00,869
815,0.03,0.01,0.00,0.00,0.20,500,lsqr,0.01,"{'alpha': 0.2, 'max_iter': 500, 'solver': 'lsq...",0.82,0.82,0.82,0.82,0.82,0.00,870
807,0.03,0.00,0.00,0.00,0.10,500,lsqr,0.01,"{'alpha': 0.1, 'max_iter': 500, 'solver': 'lsq...",0.82,0.82,0.82,0.82,0.82,0.00,871


In [ ]:
dc_scores[str(model).split("(")[0]] = {"model": clf.best_estimator_, "score": clf.best_score_}

In [ ]:
dc_scores

{'Lasso': {'model': Lasso(alpha=0.0, selection='random', tol=0.01),
  'score': 0.8697480847516432},
 'Ridge': {'model': Ridge(alpha=0.1, max_iter=500, solver='lsqr', tol=1e-07),
  'score': 0.8697477723540181}}

### Elastic Net

In [ ]:
model = ElasticNet()

In [ ]:
model.fit(X_train, y_train)
ls_medias = cross_val_score(estimator=model, X=X_test, y = y_test, cv = 4, n_jobs=-1, scoring="r2")
np.mean(ls_medias), np.std(ls_medias)

(0.8207214370458157, 0.003712356440999629)

In [ ]:
model.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'l1_ratio': 0.5,
 'max_iter': 1000,
 'positive': False,
 'precompute': False,
 'random_state': None,
 'selection': 'cyclic',
 'tol': 0.0001,
 'warm_start': False}

In [ ]:
param_grid = {
    "alpha": [x for x in range(1, 10)] + [y/10 for y in range(10)],
    "l1_ratio": [x for x in range(1, 10)] + [y/10 for y in range(10)],
    "selection": ["cyclic", "random"]
}

In [ ]:
np.prod(list(map(len, param_grid.values())))

722

In [ ]:
%%time
clf = GridSearchCV(model, param_grid, cv=4, error_score=-1000, n_jobs=-1, scoring="r2", verbose=5)
clf.fit(X_train, y_train)
print("Best score: " + str(clf.best_score_))

Fitting 4 folds for each of 722 candidates, totalling 2888 fits
[CV 1/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.816 total time=   0.1s
[CV 2/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.822 total time=   0.1s
[CV 3/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.819 total time=   0.1s
[CV 4/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.819 total time=   0.1s
[CV 1/4] END alpha=1, l1_ratio=2, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=1, l1_ratio=2, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 3/4] END alpha=1, l1_ratio=2, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=1, l1_ratio=2, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 1/4] END alpha=1, l1_ratio=1, selection=random;, score=0.816 total time=   0.2s
[CV 1/4] END alpha=1, l1_ratio=2, selection=random;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=1, l1_ratio=1, selection=random;, score=0.822 total time=

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.344e+05, tolerance: 5.668e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV 3/4] END alpha=2, l1_ratio=3, selection=random;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=2, l1_ratio=1, selection=random;, score=0.822 total time=   0.1s
[CV 4/4] END alpha=2, l1_ratio=3, selection=random;, score=-1000.000 total time=   0.0s
[CV 1/4] END alpha=2, l1_ratio=4, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=2, l1_ratio=4, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=1, selection=random;, score=0.819 total time=   0.2s
[CV 3/4] END alpha=2, l1_ratio=4, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=4, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 1/4] END alpha=2, l1_ratio=4, selection=random;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=2, l1_ratio=4, selection=random;, score=-1000.000 total time=   0.0s
[CV 3/4] END alpha=2, l1_ratio=4, selection=random;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=4, select

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.339e+05, tolerance: 5.689e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.339e+05, tolerance: 5.689e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented 

[CV 3/4] END alpha=2, l1_ratio=5, selection=random;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=5, selection=random;, score=-1000.000 total time=   0.0s
[CV 1/4] END alpha=2, l1_ratio=6, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=2, l1_ratio=6, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 3/4] END alpha=2, l1_ratio=6, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=6, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 1/4] END alpha=2, l1_ratio=6, selection=random;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=2, l1_ratio=6, selection=random;, score=-1000.000 total time=   0.0s
[CV 3/4] END alpha=2, l1_ratio=6, selection=random;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=6, selection=random;, score=-1000.000 total time=   0.0s
[CV 1/4] END alpha=2, l1_ratio=7, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=2, l1_ratio=7

...

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.314e+05, tolerance: 5.668e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.284e+05, tolerance: 5.634e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented 

[CV 3/4] END alpha=0.9, l1_ratio=0.2, selection=random;, score=0.821 total time=   0.2s
[CV 2/4] END alpha=0.9, l1_ratio=0.3, selection=cyclic;, score=0.823 total time=   0.1s
[CV 4/4] END alpha=0.9, l1_ratio=0.2, selection=random;, score=0.821 total time=   0.3s
[CV 4/4] END alpha=0.9, l1_ratio=0.0, selection=random;, score=0.821 total time=   5.0s
[CV 2/4] END alpha=0.9, l1_ratio=0.2, selection=random;, score=0.824 total time=   0.3s
[CV 3/4] END alpha=0.9, l1_ratio=0.3, selection=cyclic;, score=0.820 total time=   0.1s
[CV 4/4] END alpha=0.9, l1_ratio=0.3, selection=cyclic;, score=0.820 total time=   0.1s
[CV 1/4] END alpha=0.9, l1_ratio=0.4, selection=cyclic;, score=0.817 total time=   0.1s
[CV 2/4] END alpha=0.9, l1_ratio=0.4, selection=cyclic;, score=0.823 total time=   0.1s
[CV 1/4] END alpha=0.9, l1_ratio=0.3, selection=random;, score=0.817 total time=   0.3s
[CV 2/4] END alpha=0.9, l1_ratio=0.3, selection=random;, score=0.823 total time=   0.2s
[CV 3/4] END alpha=0.9, l1_ratio

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:425: FitFailedWarning: 
1216 fits failed out of a total of 2888.
The score on these train-test partitions for these parameters will be set to -1000.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
152 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 732, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py", line 1144, in wrapper
    estimator._validate_params()
  File "/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py", line 637, in _validate_params
    validate_parameter_constraints(
  File "/Use

Best score: 0.8697482280345004
CPU times: user 1min 16s, sys: 19.1 s, total: 1min 35s
Wall time: 3min 37s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.899e+05, tolerance: 7.539e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


In [ ]:
summary = pd.DataFrame(clf.cv_results_)

In [ ]:
dc_scores[str(model).split("(")[0]] = {"model": clf.best_estimator_, "score": clf.best_score_}

In [ ]:
dc_scores

{'Lasso': {'model': Lasso(alpha=0.0, selection='random', tol=0.01),
  'score': 0.8697480847516432},
 'Ridge': {'model': Ridge(alpha=0.1, max_iter=500, solver='lsqr', tol=1e-07),
  'score': 0.8697477723540181},
 'ElasticNet': {'model': ElasticNet(alpha=0.0, l1_ratio=0.8, selection='random'),
  'score': 0.8697482280345004}}

In [ ]:
print("Best score: " + str(clf.best_score_))

Best score: 0.8697482280345004


In [ ]:
#mejor modelo
clf.best_estimator_

ElasticNet(alpha=0.0, l1_ratio=0.8, selection='random')

In [ ]:
#mejores parámetros
clf.best_params_

{'alpha': 0.0, 'l1_ratio': 0.8, 'selection': 'random'}